In [24]:
import pandas as pd
import statsmodels.api as sm
from utils.ff_functions import ff_run_regression
from utils.ff_functions import create_coef_table
from utils.ff_functions import summarise_table
from utils.ff_functions import run_GRS

# ADD MARKDOWN HERE
FF5, FF3 and CAPM time-series regressions

Reference :
https://www.sciencedirect.com/science/article/pii/S0304405X14002323

In [25]:
PATH = '../data/processed'

# Loading processed factor df
ff5 = pd.read_parquet(f'{PATH}/ff5_factors_monthly.parquet')

# Loading processed 25 portfolio dfs
p_SIZE = pd.read_parquet(f'{PATH}/ff_portfolios_25_ME_BEME.parquet')
p_INV = pd.read_parquet(f'{PATH}/ff_portfolios_25_ME_INV.parquet')
p_OP = pd.read_parquet(f'{PATH}/ff_portfolios_25_ME_OP.parquet')

# Filtering date to year range
ff5 = ff5[ff5['Date'].between('1963-07-01', '2013-12-31')].copy()
p_SIZE = p_SIZE[p_SIZE['Date'].between('1963-07-01', '2013-12-31')].copy()
p_INV = p_INV[p_INV['Date'].between('1963-07-01', '2013-12-31')].copy()
p_OP = p_OP[p_OP['Date'].between('1963-07-01', '2013-12-31')].copy()

# Robustness check
assert len(ff5) == len(p_SIZE), 'Observation sizes do not match'
assert len(ff5) == len(p_INV), 'Observation sizes do not match'
assert len(ff5) == len(p_OP), 'Observation sizes do not match'

# Creating list from columns except Date
ff5_cols = [c for c in ff5.columns if c != 'Date']
p_SIZE_cols = [c for c in p_SIZE.columns if c != 'Date']
p_INV_cols = [c for c in p_INV.columns if c != 'Date']
p_OP_cols = [c for c in p_OP.columns if c != 'Date']

# Applying numeric transformations
ff5[ff5_cols] = ff5[ff5_cols].apply(pd.to_numeric, errors='coerce')
p_SIZE[p_SIZE_cols] = p_SIZE[p_SIZE_cols].apply(pd.to_numeric, errors='coerce')
p_INV[p_INV_cols] = p_INV[p_INV_cols].apply(pd.to_numeric, errors='coerce')
p_OP[p_OP_cols] = p_OP[p_OP_cols].apply(pd.to_numeric, errors='coerce')

# Checking for pseudo-missing values
for port in [p_SIZE, p_INV, p_OP]:
    print(port[port.isin([-999, -99.99]).any(axis=1)])

Empty DataFrame
Columns: [Date, SMALL LoBM, ME1 BM2, ME1 BM3, ME1 BM4, SMALL HiBM, ME2 BM1, ME2 BM2, ME2 BM3, ME2 BM4, ME2 BM5, ME3 BM1, ME3 BM2, ME3 BM3, ME3 BM4, ME3 BM5, ME4 BM1, ME4 BM2, ME4 BM3, ME4 BM4, ME4 BM5, BIG LoBM, ME5 BM2, ME5 BM3, ME5 BM4, BIG HiBM]
Index: []

[0 rows x 26 columns]
Empty DataFrame
Columns: [Date, SMALL LoINV, ME1 INV2, ME1 INV3, ME1 INV4, SMALL HiINV, ME2 INV1, ME2 INV2, ME2 INV3, ME2 INV4, ME2 INV5, ME3 INV1, ME3 INV2, ME3 INV3, ME3 INV4, ME3 INV5, ME4 INV1, ME4 INV2, ME4 INV3, ME4 INV4, ME4 INV5, BIG LoINV, ME5 INV2, ME5 INV3, ME5 INV4, BIG HiINV]
Index: []

[0 rows x 26 columns]
Empty DataFrame
Columns: [Date, SMALL LoOP, ME1 OP2, ME1 OP3, ME1 OP4, SMALL HiOP, ME2 OP1, ME2 OP2, ME2 OP3, ME2 OP4, ME2 OP5, ME3 OP1, ME3 OP2, ME3 OP3, ME3 OP4, ME3 OP5, ME4 OP1, ME4 OP2, ME4 OP3, ME4 OP4, ME4 OP5, BIG LoOP, ME5 OP2, ME5 OP3, ME5 OP4, BIG HiOP]
Index: []

[0 rows x 26 columns]


# Merging and Transforming

In [26]:
# Merging all portfolio dfs with factors
df_SIZE = p_SIZE.merge(
    ff5[["Date", "Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]],
    on="Date",
    how="inner"
)

df_INV = p_INV.merge(
    ff5[["Date", "Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]],
    on="Date",
    how="inner"
)

df_OP = p_OP.merge(
    ff5[["Date", "Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]],
    on="Date",
    how="inner"
)

# Converting to excess returns
size_cols = [c for c in p_SIZE.columns if c != "Date"]
inv_cols = [c for c in p_INV.columns if c != "Date"]
op_cols = [c for c in p_OP.columns if c !='Date']

df_SIZE[size_cols] = df_SIZE[size_cols].sub(df_SIZE["RF"], axis=0)
df_INV[inv_cols] = df_INV[inv_cols].sub(df_INV["RF"], axis=0)
df_OP[op_cols] = df_OP[op_cols].sub(df_OP["RF"], axis=0)

# CAPM regressions
For each of the three sets of 25 portfolios the regression equation is:
$$
\begin{equation}
R_{i,t} - R_{f,t}
=
\alpha_i
+
\beta_{i,M}\left(R_{M,t} - R_{f,t}\right)
+
\varepsilon_{i,t}
\end{equation}
$$

Where:

$$
\begin{aligned}
R_{i,t} &:\ \text{return on portfolio } i \text{ at time } t \\
R_{f,t} &:\ \text{risk-free rate} \\
R_{M,t} - R_{f,t} &:\ \text{market excess return (MKT--RF)} \\
\alpha_i &:\ \text{pricing error (abnormal return)} \\
\varepsilon_{i,t} &:\ \text{regression residual}
\end{aligned}
$$

In [27]:
# Factors that will be used in al regressions
factors_capm = ["Mkt-RF"]

port_frames = {
    "SIZE" : df_SIZE,
    "INV" : df_INV,
    "OP" : df_OP,
}

results = {}
tables = {}
summaries = {}

for idx, frame in port_frames.items():
    results[idx] = ff_run_regression(frame, factors=factors_capm)
    tables[idx] = create_coef_table(results[idx], factors=factors_capm)
    summaries[idx] = summarise_table(tables[idx])

# Unpacking
size_table, inv_table, op_table = tables["SIZE"], tables["INV"], tables["OP"]
size_summary, inv_summary, op_summary = summaries["SIZE"], summaries["INV"], summaries["OP"]

# Creating table of summaries
capm_summary_df = pd.DataFrame({
    "SIZE": size_summary,
    "INV": inv_summary,
    "OP": op_summary
}).T

display(capm_summary_df)

,mean(|alpha|),% sig alpha (<0.05),mean(R2),n_portfolios
SIZE,0.274525,44.827586,0.657457,29.0
INV,0.279179,62.068966,0.688879,29.0
OP,0.225813,44.827586,0.693559,29.0


# FF3 regressions
For each of the three sets of 25 portfolios the regression equation is:
$$
\begin{equation}
R_{i,t} - R_{f,t}
=
\alpha_i
+
\beta_{i,M}\left(R_{M,t} - R_{f,t}\right)
+
\beta_{i,S}\,\mathrm{SMB}_t
+
\beta_{i,H}\,\mathrm{HML}_t
+
\varepsilon_{i,t}
\end{equation}
$$

Where:

$$
\begin{aligned}
R_{i,t} &:\ \text{return on portfolio } i \text{ at time } t \\
R_{f,t} &:\ \text{risk-free rate} \\
R_{M,t} - R_{f,t} &:\ \text{market excess return (MKT--RF)} \\
\mathrm{SMB}_t &:\ \text{size factor (Small Minus Big)} \\
\mathrm{HML}_t &:\ \text{value factor (High Minus Low)} \\
\alpha_i &:\ \text{pricing error (abnormal return)} \\
\varepsilon_{i,t} &:\ \text{regression residual}
\end{aligned}
$$

In [28]:
# Factors that will be used in al regressions
factors_ff3 = ["Mkt-RF", "SMB", "HML"]

port_frames = {
    "SIZE" : df_SIZE,
    "INV" : df_INV,
    "OP" : df_OP,
}

results = {}
tables = {}
summaries = {}

for idx, frame in port_frames.items():
    results[idx] = ff_run_regression(frame, factors=factors_ff3)
    tables[idx] = create_coef_table(results[idx], factors=factors_ff3)
    summaries[idx] = summarise_table(tables[idx])

# Unpacking
size_table, inv_table, op_table = tables["SIZE"], tables["INV"], tables["OP"]
size_summary, inv_summary, op_summary = summaries["SIZE"], summaries["INV"], summaries["OP"]

# Creating table of all summaries
ff3_summary_df = pd.DataFrame({
    "SIZE": size_summary,
    "INV": inv_summary,
    "OP": op_summary
}).T

display(ff3_summary_df)

,mean(|alpha|),% sig alpha (<0.05),mean(R2),n_portfolios
SIZE,0.109567,25.925926,0.871746,27.0
INV,0.120320,33.333333,0.876936,27.0
OP,0.116405,29.629630,0.865416,27.0


# FF5 Regressions
We run FF5 regression on every portfolio set. The regression equation is:
$$
\begin{equation}
R_{i,t} - R_{f,t}
=
\alpha_i
+
\beta_{i,M}\left(R_{M,t} - R_{f,t}\right)
+
\beta_{i,S}\,\mathrm{SMB}_t
+
\beta_{i,H}\,\mathrm{HML}_t
+
\beta_{i,R}\, \mathrm{RMW}_t
+
\beta_{i,C}\, \mathrm{CMA}_t
+
\varepsilon_{i,t}
\end{equation}
$$

In [29]:
# Factors that will be used in al regressions
factors_ff5 = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]

port_frames = {
    "SIZE" : df_SIZE,
    "INV" : df_INV,
    "OP" : df_OP,
}

results = {}
tables = {}
summaries = {}

for idx, frame in port_frames.items():
    results[idx] = ff_run_regression(frame, factors=factors_ff5)
    tables[idx] = create_coef_table(results[idx], factors=factors_ff5)
    summaries[idx] = summarise_table(tables[idx])

# Unpacking
size_table, inv_table, op_table = tables["SIZE"], tables["INV"], tables["OP"]
size_summary, inv_summary, op_summary = summaries["SIZE"], summaries["INV"], summaries["OP"]

# Creating table of all summaries
ff5_summary_df = pd.DataFrame({
    "SIZE": size_summary,
    "INV": inv_summary,
    "OP": op_summary
}).T

display(ff5_summary_df)

,mean(|alpha|),% sig alpha (<0.05),mean(R2),n_portfolios
SIZE,0.088600,32.0,0.918923,25.0
INV,0.082392,12.0,0.931465,25.0
OP,0.071284,8.0,0.929778,25.0


# GRS Test

In [30]:
# Factors for FF5 GRS
facts_ff5 = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
# Portfolio dfs
df_dict = {"SIZE" : df_SIZE,
           "INV" : df_INV,
           "OP" : df_OP}

grs_rows = []

for name, df in df_dict.items():
    # Exclusion set - everything except Date, RF and factors
    exclude = set(["Date", "RF"] + facts_ff5)
    portfolio_cols = [c for c in df.columns if c not in exclude]

    F_stat, pVal, _, _ = run_GRS(df, portfolio_cols, facts_ff5)

    grs_rows.append({
        "set": name,
        "GRS_F": F_stat,
        "GRS_p": pVal,
        "n_portfolios": len(portfolio_cols),
    })

grs_df = pd.DataFrame(grs_rows).set_index("set")
grs_df

,GRS_F,GRS_p,n_portfolios
set,,,
SIZE,[[2.8914258962474566]],[[4.661953869278257e-06]],25
INV,[[3.4204595156324507]],[[7.135413571113247e-08]],25
OP,[[2.1993723402914753]],[[0.0007592137743297478]],25


# Exporting Processed Dataframes

In [32]:
# Exporting portfolio with factor merge - date filtered
df_SIZE.to_parquet("../data/processed/df_SIZE_ff5.parquet")
df_INV.to_parquet("../data/processed/df_INV_ff5.parquet")
df_OP.to_parquet("../data/processed/df_OP_ff5.parquet")